In [0]:
%run ../../config/utils

In [0]:
import sys
sys.path.append("..")
sys.path.append("../..")

from lib.job_manager import load_config, split_config
from lib_etl.s3 import etl_input_data_validator
import lib_etl.validations_ETL as validations
import pyspark.sql.functions as f

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)

## Load source data

In [0]:
df = spark.read.parquet(acq_member_geo).withColumn('file_modification_time', f.col('_metadata.file_modification_time'))
df.createOrReplaceTempView('df')

## Save to delta table

In [0]:
spark.sql(f"""
    INSERT OVERWRITE {bronze_member_geo_archive}
    SELECT
        merkle_id,
        mbr_sid,
        individual_id,
        household_id,
        address_id,
        zip,
        dsf_confirm_flag,
        dsf_cmra_flag,
        dsf_delivery_type,
        dsf_residence,
        dsf_business,
        dsf_drop_flag,
        dsf_drop_count,
        dsf_throwback,
        dsf_seasonal,
        dsf_vacant,
        fips_state_code,
        fips_country_code,
        usps_address_type,
        apartment_flag,
        time_zone,
        dma_code,
        geo_census_2010_tract,
        geo_census_2010_block_group,
        geo_msa_code,
        geo_lat_or_long_level,
        geo_latitude,
        geo_longitude,
        date,
        day,
        month,
        year,
        file_modification_time
    FROM df
""")